# 01 — Build cell-type prototypes

This notebook builds **cell-type reference prototypes** from a breast cancer scRNA-seq `.h5ad` reference and aligns them to the Xenium breast cancer gene panel.

**Inputs**
- `data/breast_reference.h5ad` — scRNA-seq reference AnnData object
- `data/gene_panel.json` — Xenium gene panel from the 10x Genomics Xenium dataset

**Outputs**
- `outputs/prototypes.csv` — mean expression per cell type
- `outputs/prototypes_normalized.csv` — L2-normalized prototypes for cosine similarity
- `outputs/shared_genes.txt` — genes shared between scRNA-seq and Xenium
- `outputs/prototype_reference_summary.csv` — reference cell-type counts and gene-space summary


In [31]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from sklearn.preprocessing import normalize

## Config

In [32]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

H5AD_PATH = DATA_DIR / "breast_reference.h5ad"
GENE_PANEL_PATH = DATA_DIR / "gene_panel.json"

PROTOTYPES_PATH = OUTPUT_DIR / "prototypes.csv"
PROTOTYPES_NORM_PATH = OUTPUT_DIR / "prototypes_normalized.csv"
SHARED_GENES_PATH = OUTPUT_DIR / "shared_genes.txt"
REFERENCE_SUMMARY_PATH = OUTPUT_DIR / "prototype_reference_summary.csv"

CELL_TYPE_COL = "cell_type"
GENE_SYMBOL_COL = "feature_name"

APPLY_MIN_CELL_FILTER = True
MIN_CELLS_PER_TYPE = 1000

EXCLUDE_LABELS = {
    "unknown", "Unknown",
    "unclassified", "Unclassified",
    "unclear", "Unclear",
    "nan", "None", "NA",
    "not applicable",
    "dropped",
    "ambiguous",
}

## Helper functions

In [33]:
def require_file(path: Path, description: str):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            f"Check that the file exists and the path is correct."
        )


def clean_gene_names(genes):
    return sorted(set(str(g).strip() for g in genes if pd.notna(g) and str(g).strip()))


def extract_xenium_genes_from_gene_panel(gene_panel_path: Path):
    require_file(gene_panel_path, "Xenium gene_panel.json")

    with open(gene_panel_path, "r") as f:
        gene_panel = json.load(f)

    if "payload" not in gene_panel or "targets" not in gene_panel["payload"]:
        raise ValueError(
            "Could not find gene_panel['payload']['targets']. "
            f"Top-level keys: {list(gene_panel.keys())}"
        )

    targets = gene_panel["payload"]["targets"]
    xenium_genes = []

    for target in targets:
        gene_name = None

        if (
            isinstance(target, dict)
            and "type" in target
            and isinstance(target["type"], dict)
            and "data" in target["type"]
            and isinstance(target["type"]["data"], dict)
            and "name" in target["type"]["data"]
        ):
            gene_name = target["type"]["data"]["name"]

        elif isinstance(target, dict) and "gene" in target:
            gene_name = target["gene"]

        elif isinstance(target, dict) and "name" in target:
            gene_name = target["name"]

        if gene_name is not None:
            xenium_genes.append(str(gene_name).strip())

    xenium_genes = sorted(set(g for g in xenium_genes if g))

    if len(xenium_genes) == 0:
        raise ValueError("No Xenium genes were extracted from gene_panel.json.")

    return xenium_genes


def filter_uninformative_labels(adata, cell_type_col, exclude_labels):
    labels = adata.obs[cell_type_col].astype(str)
    mask = ~labels.isin(exclude_labels)

    removed = labels[~mask].value_counts()
    print(f"Removing {(~mask).sum():,} cells with uninformative labels.")

    if len(removed) > 0:
        print("\nRemoved label counts:")
        print(removed)

    return adata[mask].copy()


def save_gene_list(genes, path: Path):
    with open(path, "w") as f:
        for gene in genes:
            f.write(gene + "\n")

## 1 · Check required inputs

In [34]:
require_file(H5AD_PATH, "breast_reference.h5ad")
require_file(GENE_PANEL_PATH, "Xenium gene_panel.json")

print("Found input files:")
print(f"H5AD reference: {H5AD_PATH}")
print(f"Gene panel:     {GENE_PANEL_PATH}")

Found input files:
H5AD reference: ../data/breast_reference.h5ad
Gene panel:     ../data/gene_panel.json


## 2 · Load Xenium gene panel

In [35]:
xenium_genes = extract_xenium_genes_from_gene_panel(GENE_PANEL_PATH)

print(f"Xenium panel genes: {len(xenium_genes):,}")
print("First 20 Xenium genes:")
print(xenium_genes[:20])

Xenium panel genes: 8,456
First 20 Xenium genes:
['A2ML1', 'AAMP', 'AAR2', 'AARSD1', 'ABAT', 'ABCA1', 'ABCA10', 'ABCA3', 'ABCA4', 'ABCA7', 'ABCA8', 'ABCB1', 'ABCB4', 'ABCB6', 'ABCC1', 'ABCC12', 'ABCC2', 'ABCC3', 'ABCC4', 'ABCC6']


## 3 · Load and inspect scRNA-seq reference

In [36]:
adata = ad.read_h5ad(H5AD_PATH)

print(adata)

print("\nCell metadata columns:")
print(adata.obs.columns.tolist())

print("\nFirst few rows of adata.obs:")
display(adata.obs.head())

print("\nGene metadata columns:")
print(adata.var.columns.tolist())

print("\nFirst few rows of adata.var:")
display(adata.var.head())

AnnData object with n_obs × n_vars = 104032 × 37389
    obs: 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'grade', 'author_cell_type', 'batch', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_rpca', 'X_umap'

Cell metadata columns:
['tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'develo

,tissue_ontology_term_id,tissue_type,assay_ontology_term_id,disease_ontology_term_id,cell_type_ontology_term_id,self_reported_ethnicity_ontology_term_id,development_stage_ontology_term_id,sex_ontology_term_id,donor_id,suspension_type,...,batch,is_primary_data,cell_type,assay,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
index,,,,,,,,,,,,,,,,,,,,,
wu_natgen__CID3586_AAGACCTCAGCATGAG,UBERON:0000310,tissue,EFO:0011025,MONDO:0004953,CL:0002543,unknown,HsapDv:0000137,PATO:0000383,wu_natgen_CID3586,cell,...,wu_natgen_2021,False,vein endothelial cell,10x 5' v1,invasive ductal breast carcinoma,female,breast,unknown,43-year-old stage,+kg8Sados1
wu_natgen__CID3586_AAGGTTCGTAGTACCT,UBERON:0000310,tissue,EFO:0011025,MONDO:0004953,CL:0002139,unknown,HsapDv:0000137,PATO:0000383,wu_natgen_CID3586,cell,...,wu_natgen_2021,False,endothelial cell of vascular tree,10x 5' v1,invasive ductal breast carcinoma,female,breast,unknown,43-year-old stage,h<HWZST!>A
wu_natgen__CID3586_ACCAGTAGTTGTGGCC,UBERON:0000310,tissue,EFO:0011025,MONDO:0004953,CL:0002139,unknown,HsapDv:0000137,PATO:0000383,wu_natgen_CID3586,cell,...,wu_natgen_2021,False,endothelial cell of vascular tree,10x 5' v1,invasive ductal breast carcinoma,female,breast,unknown,43-year-old stage,R@>-1C4K7_
wu_natgen__CID3586_ACCCACTAGATGTCGG,UBERON:0000310,tissue,EFO:0011025,MONDO:0004953,CL:0002139,unknown,HsapDv:0000137,PATO:0000383,wu_natgen_CID3586,cell,...,wu_natgen_2021,False,endothelial cell of vascular tree,10x 5' v1,invasive ductal breast carcinoma,female,breast,unknown,43-year-old stage,lEW!op2q&*
wu_natgen__CID3586_ACTGATGGTCAACTGT,UBERON:0000310,tissue,EFO:0011025,MONDO:0004953,CL:0002139,unknown,HsapDv:0000137,PATO:0000383,wu_natgen_CID3586,cell,...,wu_natgen_2021,False,endothelial cell of vascular tree,10x 5' v1,invasive ductal breast carcinoma,female,breast,unknown,43-year-old stage,5)#_aH^um)



Gene metadata columns:
['feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type']

First few rows of adata.var:


,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000286448,False,ENSG00000286448,NCBITaxon:9606,gene,728,lncRNA
ENSG00000225880,False,LINC00115,NCBITaxon:9606,gene,874,lncRNA
ENSG00000230368,False,FAM41C,NCBITaxon:9606,gene,919,lncRNA
ENSG00000187634,False,SAMD11,NCBITaxon:9606,gene,1731,protein_coding
ENSG00000188976,False,NOC2L,NCBITaxon:9606,gene,1244,protein_coding


## 4 · Validate cell-type and gene-symbol columns

In [37]:
if CELL_TYPE_COL not in adata.obs.columns:
    raise ValueError(
        f"CELL_TYPE_COL='{CELL_TYPE_COL}' was not found in adata.obs.\n\n"
        f"Available columns:\n{adata.obs.columns.tolist()}\n\n"
        "Pick the column that contains cell type labels and update CELL_TYPE_COL."
    )

if GENE_SYMBOL_COL not in adata.var.columns:
    raise ValueError(
        f"GENE_SYMBOL_COL='{GENE_SYMBOL_COL}' was not found in adata.var.\n\n"
        f"Available columns:\n{adata.var.columns.tolist()}\n\n"
        "Pick the column that contains gene symbols and update GENE_SYMBOL_COL."
    )

print(f"Using cell type column: {CELL_TYPE_COL}")
print("\nCell type counts:")
print(adata.obs[CELL_TYPE_COL].astype(str).value_counts())

print(f"\nUsing gene symbol column: {GENE_SYMBOL_COL}")
print("First 20 gene symbols:")
print(adata.var[GENE_SYMBOL_COL].astype(str).head(20).tolist())

Using cell type column: cell_type

Cell type counts:
cell_type
fibroblast                                45702
endothelial cell                          11927
vein endothelial cell                     10318
vascular associated smooth muscle cell     9172
endothelial cell of vascular tree          8749
myofibroblast cell                         6689
pericyte                                   5870
endothelial cell of artery                 4108
capillary endothelial cell                  916
cycling stromal cell                        581
Name: count, dtype: int64

Using gene symbol column: feature_name
First 20 gene symbols:
['ENSG00000286448', 'LINC00115', 'FAM41C', 'SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1', 'PERM1', 'HES4', 'ISG15', 'AGRN', 'RNF223', 'C1orf159', 'TTLL10', 'TNFRSF18', 'TNFRSF4', 'SDF4', 'B3GALT6', 'UBE2J2', 'SCNN1D']


## 5 · Remove ambiguous labels and optionally rare cell types

In [38]:
adata = filter_uninformative_labels(
    adata=adata,
    cell_type_col=CELL_TYPE_COL,
    exclude_labels=EXCLUDE_LABELS,
)

print("\nAfter label filtering:")
print(adata)
print("\nRemaining cell type counts:")
print(adata.obs[CELL_TYPE_COL].astype(str).value_counts())

Removing 0 cells with uninformative labels.

After label filtering:
AnnData object with n_obs × n_vars = 104032 × 37389
    obs: 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'grade', 'author_cell_type', 'batch', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_rpca', 'X_umap'

Remaining cell type counts:
cell_type
fibroblast                                45702
endothelial cell                          11927

In [39]:
if APPLY_MIN_CELL_FILTER:
    cell_counts = adata.obs[CELL_TYPE_COL].astype(str).value_counts()
    valid_types = cell_counts[cell_counts >= MIN_CELLS_PER_TYPE].index

    adata = adata[adata.obs[CELL_TYPE_COL].astype(str).isin(valid_types)].copy()

    print(f"Kept {len(valid_types)} cell types with >= {MIN_CELLS_PER_TYPE:,} cells.")
    print("\nRemaining cell type counts:")
    print(adata.obs[CELL_TYPE_COL].astype(str).value_counts())
else:
    print("Rare cell-type filtering skipped.")

Kept 8 cell types with >= 1,000 cells.

Remaining cell type counts:
cell_type
fibroblast                                45702
endothelial cell                          11927
vein endothelial cell                     10318
vascular associated smooth muscle cell     9172
endothelial cell of vascular tree          8749
myofibroblast cell                         6689
pericyte                                   5870
endothelial cell of artery                 4108
Name: count, dtype: int64


## 6 · Find shared genes with the Xenium panel

In [40]:
scrna_gene_symbols = clean_gene_names(adata.var[GENE_SYMBOL_COL].astype(str))
xenium_gene_set = set(xenium_genes)

shared_genes = sorted(set(scrna_gene_symbols) & xenium_gene_set)

print(f"scRNA-seq genes : {len(scrna_gene_symbols):,}")
print(f"Xenium panel    : {len(xenium_genes):,}")
print(f"Shared genes    : {len(shared_genes):,}")

print("\nFirst 30 shared genes:")
print(shared_genes[:30])

if len(shared_genes) < 100:
    missing_examples = sorted(list(xenium_gene_set - set(scrna_gene_symbols)))[:20]
    raise ValueError(
        f"Only {len(shared_genes)} shared genes found. "
        "This is probably too low for cosine-similarity validation.\n"
        "Check gene naming formats.\n\n"
        f"Example scRNA genes: {scrna_gene_symbols[:20]}\n"
        f"Example Xenium genes missing from scRNA: {missing_examples}"
    )

save_gene_list(shared_genes, SHARED_GENES_PATH)
print(f"\nSaved shared genes → {SHARED_GENES_PATH}")

scRNA-seq genes : 37,361
Xenium panel    : 8,456
Shared genes    : 5,054

First 30 shared genes:
['A2ML1', 'AAMP', 'AAR2', 'AARSD1', 'ABAT', 'ABCA1', 'ABCA10', 'ABCA3', 'ABCA4', 'ABCA7', 'ABCA8', 'ABCB1', 'ABCB4', 'ABCB6', 'ABCC1', 'ABCC12', 'ABCC2', 'ABCC3', 'ABCC4', 'ABCC6', 'ABCC8', 'ABCC9', 'ABCD1', 'ABCD3', 'ABCD4', 'ABCF3', 'ABCG1', 'ABCG2', 'ABHD11', 'ABHD6']

Saved shared genes → ../outputs/shared_genes.txt


## 7 · Subset expression to shared genes

In [41]:
var_gene_symbols = adata.var[GENE_SYMBOL_COL].astype(str)
shared_mask = var_gene_symbols.isin(shared_genes).values

adata_shared = adata[:, shared_mask].copy()

adata_shared.var["gene_symbol"] = adata_shared.var[GENE_SYMBOL_COL].astype(str).values
adata_shared.var_names = adata_shared.var["gene_symbol"].astype(str).values

print("Shared AnnData object:")
print(adata_shared)

X = adata_shared.X
if sp.issparse(X):
    X = X.toarray()
else:
    X = np.asarray(X)

expr_shared = pd.DataFrame(
    X,
    index=adata_shared.obs_names.astype(str),
    columns=adata_shared.var_names.astype(str),
)

if expr_shared.columns.duplicated().any():
    n_dupes = expr_shared.columns.duplicated().sum()
    print(f"Found {n_dupes} duplicated shared gene symbols. Collapsing duplicates by sum.")
    expr_shared = expr_shared.T.groupby(level=0).sum().T

expr_shared = expr_shared.reindex(columns=shared_genes, fill_value=0)

print("Shared expression matrix:", expr_shared.shape, "(cells × shared genes)")
display(expr_shared.iloc[:5, :5])

Shared AnnData object:
AnnData object with n_obs × n_vars = 102535 × 5054
    obs: 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'grade', 'author_cell_type', 'batch', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'gene_symbol'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_rpca', 'X_umap'
Shared expression matrix: (102535, 5054) (cells × shared genes)


,A2ML1,AAMP,AAR2,AARSD1,ABAT
index,,,,,
wu_natgen__CID3586_AAGACCTCAGCATGAG,0.0,1.157802,0.0,0.0,0.0
wu_natgen__CID3586_AAGGTTCGTAGTACCT,0.0,0.000000,0.0,0.0,0.0
wu_natgen__CID3586_ACCAGTAGTTGTGGCC,0.0,0.000000,0.0,0.0,0.0
wu_natgen__CID3586_ACCCACTAGATGTCGG,0.0,0.000000,0.0,0.0,0.0
wu_natgen__CID3586_ACTGATGGTCAACTGT,0.0,0.000000,0.0,0.0,0.0


## 8 · Build prototypes by averaging per cell type

In [42]:
cell_types = adata.obs[CELL_TYPE_COL].astype(str)
cell_types = cell_types.loc[expr_shared.index]

prototypes = expr_shared.groupby(cell_types).mean()

print("Prototype matrix:", prototypes.shape, "(cell types × genes)")
print("\nCell types:")
print(prototypes.index.tolist())

display(prototypes.iloc[:5, :5])

Prototype matrix: (8, 5054) (cell types × genes)

Cell types:
['endothelial cell', 'endothelial cell of artery', 'endothelial cell of vascular tree', 'fibroblast', 'myofibroblast cell', 'pericyte', 'vascular associated smooth muscle cell', 'vein endothelial cell']


,A2ML1,AAMP,AAR2,AARSD1,ABAT
cell_type,,,,,
endothelial cell,0.001064,0.247868,0.083641,0.055516,0.008483
endothelial cell of artery,0.000533,0.228971,0.093853,0.044692,0.005978
endothelial cell of vascular tree,0.001511,0.109087,0.027541,0.039811,0.003689
fibroblast,0.000456,0.210832,0.068810,0.037553,0.013576
myofibroblast cell,0.003495,0.123415,0.036209,0.034725,0.004832


## 9 · L2-normalize for cosine similarity

In [43]:
proto_norm = pd.DataFrame(
    normalize(prototypes.values, norm="l2", axis=1),
    index=prototypes.index,
    columns=prototypes.columns,
)

print("Normalized prototype matrix:", proto_norm.shape)
display(proto_norm.iloc[:5, :5])

Normalized prototype matrix: (8, 5054)


,A2ML1,AAMP,AAR2,AARSD1,ABAT
cell_type,,,,,
endothelial cell,0.000055,0.012799,0.004319,0.002867,0.000438
endothelial cell of artery,0.000026,0.011032,0.004522,0.002153,0.000288
endothelial cell of vascular tree,0.000153,0.011066,0.002794,0.004039,0.000374
fibroblast,0.000028,0.012987,0.004239,0.002313,0.000836
myofibroblast cell,0.000359,0.012681,0.003721,0.003568,0.000496


## 10 · Save outputs

In [44]:
prototypes.to_csv(PROTOTYPES_PATH)
proto_norm.to_csv(PROTOTYPES_NORM_PATH)

summary = (
    adata.obs[CELL_TYPE_COL]
    .astype(str)
    .value_counts()
    .rename_axis("cell_type")
    .reset_index(name="n_cells")
)

summary["n_shared_genes"] = len(shared_genes)
summary["n_xenium_genes"] = len(xenium_genes)
summary["n_scrna_genes"] = len(scrna_gene_symbols)
summary["min_cells_filter_applied"] = APPLY_MIN_CELL_FILTER
summary["min_cells_per_type"] = MIN_CELLS_PER_TYPE if APPLY_MIN_CELL_FILTER else None

summary.to_csv(REFERENCE_SUMMARY_PATH, index=False)

print("Saved:")
print(f"Prototypes:             {PROTOTYPES_PATH}")
print(f"Normalized prototypes:  {PROTOTYPES_NORM_PATH}")
print(f"Shared genes:           {SHARED_GENES_PATH}")
print(f"Reference summary:      {REFERENCE_SUMMARY_PATH}")

Saved:
Prototypes:             ../outputs/prototypes.csv
Normalized prototypes:  ../outputs/prototypes_normalized.csv
Shared genes:           ../outputs/shared_genes.txt
Reference summary:      ../outputs/prototype_reference_summary.csv


## 11 · Final sanity checks

In [45]:
print("Final sanity check")
print("------------------")
print(f"Reference cells:     {adata.n_obs:,}")
print(f"scRNA-seq genes:     {len(scrna_gene_symbols):,}")
print(f"Xenium genes:        {len(xenium_genes):,}")
print(f"Shared genes:        {len(shared_genes):,}")
print(f"Cell types:          {prototypes.shape[0]:,}")
print(f"Prototype shape:     {prototypes.shape}")
print(f"Normalized shape:    {proto_norm.shape}")

assert prototypes.shape == proto_norm.shape
assert list(prototypes.columns) == shared_genes
assert list(proto_norm.columns) == shared_genes

print("\nNotebook 01 completed successfully.")

Final sanity check
------------------
Reference cells:     102,535
scRNA-seq genes:     37,361
Xenium genes:        8,456
Shared genes:        5,054
Cell types:          8
Prototype shape:     (8, 5054)
Normalized shape:    (8, 5054)

Notebook 01 completed successfully.
